
# 🚢 Kaggle Starter: Titanic — Machine Learning from Disaster
**Autor:** Eugenia Rusu  
**Obiectiv:** Construiește rapid un baseline curat (scikit-learn) + fișier `submission.csv` gata de urcat pe Kaggle.  
**Link competiție:** https://www.kaggle.com/c/titanic  

## Ce conține notebook-ul
- Încărcarea datelor (train/test)  
- EDA de bază (verificare lipsuri, distribuții)  
- Feature Engineering minim (sex, vârstă imputată, cabină/port simplificat)  
- Model de clasificare (Logistic Regression + RandomForest opțional)  
- Cross-validation corectă (StratifiedKFold)  
- Export `submission.csv`



## 1. Setup & Pachete
Rulează celula pentru a instala/importa pachetele necesare.  
> Dacă rulezi pe Kaggle Notebooks, multe pachete sunt deja instalate.


In [ ]:

# !pip install pandas numpy scikit-learn matplotlib --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
print("Pachete încărcate ✅")



## 2. Încărcarea datelor
- Dacă rulezi local, setează variabilele spre fișierele `train.csv` și `test.csv`.  
- Pe Kaggle, acestea sunt disponibile în `/kaggle/input/titanic/`.


In [ ]:

# CĂI IMPLICITE (Kaggle)
TRAIN_PATH = "/kaggle/input/titanic/train.csv"
TEST_PATH  = "/kaggle/input/titanic/test.csv"

# Dacă rulezi local, setează manual:
# TRAIN_PATH = "train.csv"
# TEST_PATH  = "test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print(train.shape, test.shape)
train.head()



## 3. EDA rapid (explorare de bază)
Scop: înțelegem lipsurile și variabilele utile.  
Notă: evităm grafice complicate la început — vrem un baseline rapid.


In [ ]:

# Lipsuri pe coloană
missing = train.isna().mean().sort_values(ascending=False)
display(missing.to_frame("proportion_missing").head(10))

# Distribuții principale
print(train["Survived"].value_counts(normalize=True))
print(train[["Pclass","Survived"]].groupby("Pclass").mean())

# Mic grafic: vârsta (doar dacă există)
train["Age"].hist(bins=30)
plt.title("Distribuția vârstei (train)")
plt.xlabel("Age"); plt.ylabel("Count")
plt.show()



## 4. Feature Engineering minim
- Imputăm vârsta cu mediană.  
- Codăm sexul și portul.  
- Folosim câteva coloane robuste pentru un baseline curat.


In [ ]:

TARGET = "Survived"
ID_COL = "PassengerId"

features_num = ["Age", "SibSp", "Parch", "Fare"]
features_cat = ["Pclass", "Sex", "Embarked"]  # Pclass o tratăm ca categorical

X = train[features_num + features_cat].copy()
y = train[TARGET].copy()
X_test = test[features_num + features_cat].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_num),
        ("cat", categorical_transformer, features_cat),
    ]
)

model = LogisticRegression(max_iter=1000, n_jobs=None)

clf = Pipeline(steps=[("preprocess", preprocess),
                     ("model", model)])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")
print("CV Accuracy (mean ± std):", cv_scores.mean().round(4), "±", cv_scores.std().round(4))



## 5. Antrenare finală & creare `submission.csv`
După validarea încrucișată, antrenăm pe tot `train` și prezicem `test`.


In [ ]:

clf.fit(X, y)
preds = clf.predict(X_test)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: preds.astype(int)
})
submission_path = "submission_titanic_baseline.csv"
submission.to_csv(submission_path, index=False)
print(f"Am salvat: {submission_path}")
submission.head()



## 6. (Opțional) Îmbunătățire rapidă: RandomForest
Schimbă modelul din pipeline pentru a testa un algoritm non-liniar.


In [ ]:

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

clf_rf = Pipeline(steps=[("preprocess", preprocess),
                        ("model", rf)])

cv_scores_rf = cross_val_score(clf_rf, X, y, cv=skf, scoring="accuracy")
print("RF CV Accuracy:", cv_scores_rf.mean().round(4), "±", cv_scores_rf.std().round(4))

clf_rf.fit(X, y)
preds_rf = clf_rf.predict(X_test)
submission_rf = pd.DataFrame({ID_COL: test[ID_COL], TARGET: preds_rf.astype(int)})
submission_path_rf = "submission_titanic_rf.csv"
submission_rf.to_csv(submission_path_rf, index=False)
print(f"Am salvat: {submission_path_rf}")
submission_rf.head()
